In [1]:
import pandas as pd

df = pd.read_csv('history.csv')

colunas = df.columns.tolist()
print(colunas)

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'CustomerFeedback', 'MonthlyIncome']


In [2]:
listaNomesTraduzidos = [
    'id_cliente',
    'genero',
    'idoso',
    'possui_companheiro',
    'possui_dependentes',
    'tempo_permanencia',
    'servico_telefonico',
    'multiplas_linhas_telefonicas',
    'servico_internet',
    'seguranca_online',
    'backup_online',
    'protecao_dispositivos',
    'suporte_tecnico',
    'streaming_tv',
    'streaming_filmes',
    'tipo_contrato',
    'faturamento_digital',
    'metodo_pagamento',
    'cobranca_mensal',
    'cobranca_total_acumulada',
    'cancelamento_servico',
    'feedback_cliente',
    'renda_mensal'
]
df.columns = listaNomesTraduzidos

In [3]:
## Descomentar se quiser salvar o arquivo com os nomes das colunas traduzidos
#df.to_csv('history_traduzido.csv', index=False, encoding='utf-8')

In [ ]:

colunas_inteiras = {'idoso', 'tempo_permanencia'}
colunas_monetarias = {
    'cobranca_mensal',
    'cobranca_total_acumulada',
    'renda_mensal'
}

for coluna in df.columns:
    if coluna in colunas_inteiras:
        # Int64 aceita valores ausentes; textos como "unknown" viram <NA>.
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce').astype('Int64')
    elif coluna in colunas_monetarias:
        # Arredonda valores monetários para duas casas decimais.
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce').round(2).astype('Float64')
    else:
        df[coluna] = df[coluna].astype('string')

print('Tipos das colunas:')
print(df.dtypes)


df.to_csv(
    'history_traduzido.csv',
    index=False,
    encoding='utf-8',
    float_format='%.2f'
)

Tipos das colunas:
id_cliente                       string
genero                           string
idoso                             Int64
possui_companheiro               string
possui_dependentes               string
tempo_permanencia                 Int64
servico_telefonico               string
multiplas_linhas_telefonicas     string
servico_internet                 string
seguranca_online                 string
backup_online                    string
protecao_dispositivos            string
suporte_tecnico                  string
streaming_tv                     string
streaming_filmes                 string
tipo_contrato                    string
faturamento_digital              string
metodo_pagamento                 string
cobranca_mensal                 Float64
cobranca_total_acumulada        Float64
cancelamento_servico             string
feedback_cliente                 string
renda_mensal                    Float64
dtype: object


In [5]:
total_duplicadas = df.duplicated().sum()
print(f"Total de linhas duplicadas: {total_duplicadas}")
if total_duplicadas > 0:
    linhas_duplicadas = df[df.duplicated(keep=False)]
    print(linhas_duplicadas)


Total de linhas duplicadas: 0


In [ ]:
tem_nulo = df.isnull().values.any()
print(f"O arquivo possui valores nulos? {'Sim' if tem_nulo else 'Não'}\n")

if tem_nulo:
    print("--- Quantidade de nulos por coluna ---")
    print(df.isnull().sum())
    
    print("\n--- Linhas que contêm algum valor nulo ---")
    linhas_com_nulos = df[df.isnull().any(axis=1)]
    print(linhas_com_nulos)

O arquivo possui valores nulos? Sim

--- Quantidade de nulos por coluna ---
id_cliente                        0
genero                            0
idoso                             0
possui_companheiro                0
possui_dependentes                0
tempo_permanencia                19
servico_telefonico                0
multiplas_linhas_telefonicas      0
servico_internet                  0
seguranca_online                  0
backup_online                     0
protecao_dispositivos             0
suporte_tecnico                   0
streaming_tv                      0
streaming_filmes                  0
tipo_contrato                     0
faturamento_digital               0
metodo_pagamento                  0
cobranca_mensal                   0
cobranca_total_acumulada        278
cancelamento_servico              0
feedback_cliente                  0
renda_mensal                      0
dtype: int64

--- Linhas que contêm algum valor nulo ---
      id_cliente  genero  idoso possui_

In [7]:
df_colunas_traduzidas = pd.read_csv('history_traduzido.csv')

In [8]:
coluna_alvo = 'cobranca_total_acumulada'
mascara_nulos = df_colunas_traduzidas[coluna_alvo].isna()
tempo = pd.to_numeric(df_colunas_traduzidas['tempo_permanencia'], errors='coerce')
cobranca_mensal = pd.to_numeric(df_colunas_traduzidas['cobranca_mensal'], errors='coerce')
mascara_preenchiveis = mascara_nulos & tempo.notna() & cobranca_mensal.notna()

# Preenche somente valores ausentes cujos dados usados no cálculo sejam numéricos.
df_colunas_traduzidas.loc[mascara_preenchiveis, coluna_alvo] = (
    tempo.loc[mascara_preenchiveis] * cobranca_mensal.loc[mascara_preenchiveis]
)

colunas_monetarias = [
    'cobranca_mensal',
    'cobranca_total_acumulada',
    'renda_mensal'
]
for coluna in colunas_monetarias:
    df_colunas_traduzidas[coluna] = pd.to_numeric(
        df_colunas_traduzidas[coluna], errors='coerce'
    ).round(2)

print(f"Valores preenchidos: {mascara_preenchiveis.sum()}")
print(f"Valores ainda ausentes (dados de entrada inválidos ou ausentes): {df_colunas_traduzidas[coluna_alvo].isna().sum()}")
#df_colunas_traduzidas.to_csv('history_traduzido.csv', index=False, encoding='utf-8', float_format='%.2f')

Valores preenchidos: 277
Valores ainda ausentes (dados de entrada inválidos ou ausentes): 1


In [3]:
# Carregar o DataFrame
df = pd.read_csv('history.csv')

tipos_colunas = {
    'tenure': 'int64',
    
    # Decimais / Floats (Valores Financeiros)
    'MonthlyCharges': 'float64',
    'TotalCharges': 'float64',
    'MonthlyIncome': 'float64',
    
    # Texto / String (Identificador único e Textos Livres)
    'customerID': 'string',
    'CustomerFeedback': 'string',
    
    # Categóricos (Variáveis Binárias ou de Múltiplas Categorias)
    'SeniorCitizen': 'category',
    'gender': 'category',
    'Partner': 'category',
    'Dependents': 'category',
    'PhoneService': 'category',
    'MultipleLines': 'category',
    'InternetService': 'category',
    'OnlineSecurity': 'category',
    'OnlineBackup': 'category',
    'DeviceProtection': 'category',
    'TechSupport': 'category',
    'StreamingTV': 'category',
    'StreamingMovies': 'category',
    'Contract': 'category',
    'PaperlessBilling': 'category',
    'PaymentMethod': 'category',
    'Churn': 'category'
}

# Tratamento prévio de colunas numéricas (força conversão e evita erros de tipo str)
colunas_float = ['MonthlyCharges', 'TotalCharges', 'MonthlyIncome']
colunas_int = ['SeniorCitizen', 'tenure']

for col in colunas_float + colunas_int:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['SeniorCitizen'] = df['SeniorCitizen'].fillna(0)
df['tenure'] = df['tenure'].fillna(0)

df = df.astype(tipos_colunas)

print("--- Tipos de dados atualizados (Inglês) ---")
print(df.dtypes)

--- Tipos de dados atualizados (Inglês) ---
customerID            string
gender              category
SeniorCitizen       category
Partner             category
Dependents          category
tenure                 int64
PhoneService        category
MultipleLines       category
InternetService     category
OnlineSecurity      category
OnlineBackup        category
DeviceProtection    category
TechSupport         category
StreamingTV         category
StreamingMovies     category
Contract            category
PaperlessBilling    category
PaymentMethod       category
MonthlyCharges       float64
TotalCharges         float64
Churn               category
CustomerFeedback      string
MonthlyIncome        float64
dtype: object


In [ ]:
# Carregar o DataFrame
df = pd.read_csv('history_traduzido.csv')

tipos_colunas = {
    # Inteiros (Identificadores e Contadores)
    'tempo_permanencia': 'int64',
    
    # Decimais / Floats (Valores Financeiros)
    'cobranca_mensal': 'float64',
    'cobranca_total_acumulada': 'float64',
    'renda_mensal': 'float64',
    
    # Texto / String (Identificador único e Textos Livres)
    'id_cliente': 'string',
    'feedback_cliente': 'string',
    
    # Categóricos (Variáveis Binárias ou de Múltiplas Categorias)
    'idoso': 'category',
    'genero': 'category',
    'possui_companheiro': 'category',
    'possui_dependentes': 'category',
    'servico_telefonico': 'category',
    'multiplas_linhas_telefonicas': 'category',
    'servico_internet': 'category',
    'seguranca_online': 'category',
    'backup_online': 'category',
    'protecao_dispositivos': 'category',
    'suporte_tecnico': 'category',
    'streaming_tv': 'category',
    'streaming_filmes': 'category',
    'tipo_contrato': 'category',
    'faturamento_digital': 'category',
    'metodo_pagamento': 'category',
    'cancelamento_servico': 'category'
}

# Tratamento prévio de colunas numéricas (evita erros com 'str' em colunas de números)
colunas_float = ['cobranca_mensal', 'cobranca_total_acumulada', 'renda_mensal']
colunas_int = ['idoso', 'tempo_permanencia']

for col in colunas_float + colunas_int:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
df['idoso'] = df['idoso'].fillna(0)
df['tempo_permanencia'] = df['tempo_permanencia'].fillna(0)

df = df.astype(tipos_colunas)

print("--- Tipos de dados atualizados ---")
print(df.dtypes)

--- Tipos de dados atualizados ---
id_cliente                        string
genero                          category
idoso                           category
possui_companheiro              category
possui_dependentes              category
tempo_permanencia                  int64
servico_telefonico              category
multiplas_linhas_telefonicas    category
servico_internet                category
seguranca_online                category
backup_online                   category
protecao_dispositivos           category
suporte_tecnico                 category
streaming_tv                    category
streaming_filmes                category
tipo_contrato                   category
faturamento_digital             category
metodo_pagamento                category
cobranca_mensal                  float64
cobranca_total_acumulada         float64
cancelamento_servico            category
feedback_cliente                  string
renda_mensal                     float64
dtype: object
